<a href="https://colab.research.google.com/github/maryamnoureen/iomt-reliability-security/blob/main/Math_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
================================================================================
IoMT Device Reliability and Security Exposure Assessment Platform
Implementation of Neo's 16-Step IoMT Reliability & Criticality Algorithm
================================================================================
"""

import sys
import math
import random
from typing import Dict, List, Tuple, Optional


# ==============================================================================
# DATA MODELS & PROTOCOL PROFILES
# ==============================================================================

# Step 6: Predefined Security Levels
SECURITY_LEVELS = {
    "Strong": 0.9,
    "Medium": 0.7,
    "Weak": 0.4
}

PROTOCOL_PROFILES = {
    "Wired Ethernet (802.3)": {"family": "Wired", "default_p": "Strong"},
    "Isolated Serial (RS-485)": {"family": "Wired", "default_p": "Strong"},
    "Wi-Fi WPA3 (802.11ax)": {"family": "Wireless", "default_p": "Strong"},
    "Wi-Fi WPA2 (802.11n)": {"family": "Wireless", "default_p": "Medium"},
    "Bluetooth Low Energy (BLE)": {"family": "Wireless", "default_p": "Medium"},
    "Zigbee (802.15.4)": {"family": "Wireless", "default_p": "Weak"},
}


class IoMTDevice:
    """
    Represents an IoMT device in the network.
    Strict constraint: Must contain at least 2 sensors.
    """
    def __init__(
        self,
        device_id: str,
        name: str,
        protocol: str,
        sensor_readings: List[float],
        initial_reliability: float,
        frequency_hz: float,
        security_level: str = "Medium"
    ):
        # Strict Constraint Validation: >= 2 sensors
        if len(sensor_readings) < 2:
            raise ValueError(f"Constraint Violation: Device {device_id} must have >= 2 sensors (got {len(sensor_readings)}).")

        self.device_id = device_id
        self.name = name
        self.protocol = protocol
        self.protocol_family = PROTOCOL_PROFILES.get(protocol, {}).get("family", "Wireless")

        # Step 2: Store independent sensor readings (each in [0.0, 1.0])
        self.sensor_readings = [float(s) for s in sensor_readings]

        # Initial reliability R_initial in [0.0, 1.0]
        self.initial_reliability = float(initial_reliability)

        # Communication frequency f in Hz
        self.frequency_hz = float(frequency_hz)

        # Step 6: Assign security level P (Strong=0.9, Medium=0.7, Weak=0.4)
        self.security_level_name = security_level
        self.security_level_p = SECURITY_LEVELS.get(security_level, 0.7)

    @property
    def sensor_count(self) -> int:
        return len(self.sensor_readings)

    # Step 3: Calculate Sensor Reliability:
    # Rsensor = (sum of sensor values) / (number of sensors)
    def calculate_sensor_reliability(self) -> float:
        if not self.sensor_readings:
            return 0.0
        return sum(self.sensor_readings) / len(self.sensor_readings)

    # Step 4: Calculate Device Reliability:
    # Rdevice = (Rinitial + Rsensor) / 2
    def calculate_device_reliability(self) -> float:
        r_sensor = self.calculate_sensor_reliability()
        return (self.initial_reliability + r_sensor) / 2.0

    # Step 5: Calculate Normalised Communication Frequency:
    # F = f / f_max
    def calculate_normalised_frequency(self, f_max: float) -> float:
        if f_max <= 0:
            return 0.0
        return min(1.0, max(0.0, self.frequency_hz / f_max))

    # Step 7: Calculate Security Exposure:
    # E = (F + (1 - P)) / 2
    def calculate_security_exposure(self, f_max: float) -> float:
        f_norm = self.calculate_normalised_frequency(f_max)
        p_val = self.security_level_p
        return (f_norm + (1.0 - p_val)) / 2.0


# ==============================================================================
# 16-STEP ALGORITHM NETWORK ENGINE
# ==============================================================================

class IoMTAlgorithmEngine:
    """
    Implements the exact 16-step IoMT assessment algorithm.
    """
    def __init__(self):
        self.devices: Dict[str, IoMTDevice] = {}
        self.dependency_weights: Dict[Tuple[str, str], float] = {}
        self.init_default_hospital_network()

    def init_default_hospital_network(self):
        """
        Step 1 & 2: Create D1-D5 with >= 2 sensor readings each.
        """
        self.devices.clear()
        self.dependency_weights.clear()

        # D1: Infusion Pump (Wi-Fi, 2 sensors: Flow rate, Drop count)
        d1 = IoMTDevice(
            device_id="D1",
            name="Infusion Pump",
            protocol="Wi-Fi WPA2 (802.11n)",
            sensor_readings=[0.88, 0.92],
            initial_reliability=0.85,
            frequency_hz=2.0,
            security_level="Medium"   # P = 0.7
        )
        # D2: ECG Vital Monitor (Wired Ethernet, 3 sensors: Lead II, Lead V1, Pulse)
        d2 = IoMTDevice(
            device_id="D2",
            name="ECG Vital Monitor",
            protocol="Wired Ethernet (802.3)",
            sensor_readings=[0.94, 0.96, 0.91],
            initial_reliability=0.92,
            frequency_hz=10.0,
            security_level="Strong"   # P = 0.9
        )
        # D3: ICU Ventilator (Wired Serial, 2 sensors: Airway pressure, Tidal volume)
        d3 = IoMTDevice(
            device_id="D3",
            name="ICU Ventilator",
            protocol="Isolated Serial (RS-485)",
            sensor_readings=[0.95, 0.90],
            initial_reliability=0.96,
            frequency_hz=5.0,
            security_level="Strong"   # P = 0.9
        )
        # D4: Continuous Glucose Monitor (BLE, 2 sensors: Glucose, Trend delta)
        d4 = IoMTDevice(
            device_id="D4",
            name="Continuous Glucose Monitor",
            protocol="Bluetooth Low Energy (BLE)",
            sensor_readings=[0.78, 0.82],
            initial_reliability=0.80,
            frequency_hz=1.0,
            security_level="Medium"   # P = 0.7
        )
        # D5: Pulse Oximeter SpO2 (Zigbee, 2 sensors: SpO2, Perfusion index)
        d5 = IoMTDevice(
            device_id="D5",
            name="Pulse Oximeter (SpO2)",
            protocol="Zigbee (802.15.4)",
            sensor_readings=[0.72, 0.76],
            initial_reliability=0.78,
            frequency_hz=4.0,
            security_level="Weak"     # P = 0.4
        )

        for dev in [d1, d2, d3, d4, d5]:
            self.devices[dev.device_id] = dev

        # Step 8 & 9: Store device connections and dependency values (W)
        # D2 (ECG) telemetry directly feeds D1 (Infusion Pump) with weight W = 0.35
        self.set_dependency("D2", "D1", 0.35)
        # D3 (Ventilator) coordinates with D2 (ECG Monitor) with weight W = 0.25
        self.set_dependency("D3", "D2", 0.25)
        # D5 (SpO2) feeds oxygen status into D3 (Ventilator) with weight W = 0.40
        self.set_dependency("D5", "D3", 0.40)
        # D4 (Glucose Monitor) coordinates basal rate on D1 (Infusion Pump) with weight W = 0.30
        self.set_dependency("D4", "D1", 0.30)

    def set_dependency(self, source_id: str, target_id: str, weight: float):
        if weight <= 0.0:
            self.dependency_weights.pop((source_id, target_id), None)
        else:
            self.dependency_weights[(source_id, target_id)] = min(1.0, max(0.0, float(weight)))

    def get_max_frequency(self) -> float:
        if not self.devices:
            return 1.0
        return max(d.frequency_hz for d in self.devices.values())

    # Step 8: Create Adjacency Matrix
    def build_adjacency_matrix(self) -> Tuple[List[str], List[List[int]]]:
        dev_ids = sorted(self.devices.keys())
        n = len(dev_ids)
        matrix = [[0 for _ in range(n)] for _ in range(n)]
        for i, src in enumerate(dev_ids):
            for j, tgt in enumerate(dev_ids):
                if (src, tgt) in self.dependency_weights and self.dependency_weights[(src, tgt)] > 0:
                    matrix[i][j] = 1
        return dev_ids, matrix

    # Step 9: Store Dependency Value W for each relevant connection
    def build_dependency_matrix(self) -> Tuple[List[str], List[List[float]]]:
        dev_ids = sorted(self.devices.keys())
        n = len(dev_ids)
        matrix = [[0.0 for _ in range(n)] for _ in range(n)]
        for i, src in enumerate(dev_ids):
            for j, tgt in enumerate(dev_ids):
                matrix[i][j] = self.dependency_weights.get((src, tgt), 0.0)
        return dev_ids, matrix

    # Step 10: Calculate Network / Derived Edge Reliability
    # Rnetwork = (R1 + R2 + ... + Rn) / n
    def calculate_network_reliability(self, device_reliabilities: Optional[Dict[str, float]] = None) -> float:
        if device_reliabilities is None:
            device_reliabilities = {
                d_id: d.calculate_device_reliability()
                for d_id, d in self.devices.items()
            }
        if not device_reliabilities:
            return 0.0
        return sum(device_reliabilities.values()) / len(device_reliabilities)

    # Step 12 to 15: Criticality Analysis
    def run_criticality_analysis(self) -> Dict[str, any]:
        dev_ids = sorted(self.devices.keys())
        base_reliabilities = {d_id: self.devices[d_id].calculate_device_reliability() for d_id in dev_ids}

        # Step 11: Save baseline as R_old
        r_old = self.calculate_network_reliability(base_reliabilities)
        criticality_results = {}

        # Step 12: For each device D1 -> D5:
        for target_dev in dev_ids:
            # Temporarily mark device as affected (fails: R_target -> 0.0)
            affected_reliabilities = base_reliabilities.copy()
            affected_reliabilities[target_dev] = 0.0

            # Cascade: Dependent devices degrade proportional to dependency value W
            for other_dev in dev_ids:
                if other_dev != target_dev:
                    w = self.dependency_weights.get((target_dev, other_dev), 0.0)
                    if w > 0:
                        affected_reliabilities[other_dev] = affected_reliabilities[other_dev] * (1.0 - w)

            # Recalculate network reliability (R_new)
            r_new = self.calculate_network_reliability(affected_reliabilities)

            # Calculate Criticality = (R_old - R_new) / R_old
            criticality = (r_old - r_new) / r_old if r_old > 0 else 0.0

            criticality_results[target_dev] = {
                "r_old": r_old,
                "r_new": r_new,
                "delta_r": r_old - r_new,
                "criticality": criticality,
                "criticality_pct": criticality * 100.0
            }
            # Step 13: Restore device and move to next (handled naturally by base_reliabilities.copy())

        # Step 14 & 15: Compare criticality values; device with highest criticality is Critical
        most_critical_id = max(criticality_results.keys(), key=lambda k: criticality_results[k]["criticality"])

        return {
            "r_old": r_old,
            "r_old_scaled": r_old * 100.0,
            "criticality_table": criticality_results,
            "most_critical_device": most_critical_id,
            "most_critical_device_name": self.devices[most_critical_id].name,
            "max_criticality_pct": criticality_results[most_critical_id]["criticality_pct"]
        }

    # Step 16: Complete Sequential Execution
    def execute_full_algorithm(self) -> Dict[str, any]:
        dev_ids = sorted(self.devices.keys())
        f_max = self.get_max_frequency()
        device_metrics = {}

        for d_id in dev_ids:
            d = self.devices[d_id]
            r_sensor = d.calculate_sensor_reliability()
            r_device = d.calculate_device_reliability()
            f_norm = d.calculate_normalised_frequency(f_max)
            e_val = d.calculate_security_exposure(f_max)

            device_metrics[d_id] = {
                "name": d.name,
                "protocol": d.protocol,
                "family": d.protocol_family,
                "sensor_readings": d.sensor_readings,
                "sensor_count": d.sensor_count,
                "r_initial": d.initial_reliability,
                "r_sensor": r_sensor,
                "r_device": r_device,
                "frequency_hz": d.frequency_hz,
                "f_norm": f_norm,
                "security_name": d.security_level_name,
                "p_val": d.security_level_p,
                "exposure_e": e_val,
                "r_device_100": r_device * 100.0,
                "exposure_e_100": e_val * 100.0
            }

        _, adj_matrix = self.build_adjacency_matrix()
        _, dep_matrix = self.build_dependency_matrix()
        crit_results = self.run_criticality_analysis()

        return {
            "device_ids": dev_ids,
            "f_max": f_max,
            "device_metrics": device_metrics,
            "adj_matrix": adj_matrix,
            "dep_matrix": dep_matrix,
            "criticality_analysis": crit_results
        }


# ==============================================================================
# MENU-DRIVEN CONSOLE APPLICATION
# ==============================================================================

def print_header(title: str):
    print("\n" + "=" * 80)
    print(f" {title.center(78)} ")
    print("=" * 80)

def print_subheader(title: str):
    print("\n" + "-" * 80)
    print(f" >> {title}")
    print("-" * 80)

def print_table(headers: List[str], rows: List[List[str]], col_widths: List[int]):
    header_line = " | ".join(h.ljust(w) for h, w in zip(headers, col_widths))
    sep_line = "-+-".join("-" * w for w in col_widths)
    print(header_line)
    print(sep_line)
    for row in rows:
        row_line = " | ".join(str(cell).ljust(w) for cell, w in zip(row, col_widths))
        print(row_line)


class IoMTAlgorithmApp:
    def __init__(self):
        self.engine = IoMTAlgorithmEngine()

    def run(self):
        while True:
            print_header("IoMT RELIABILITY & SECURITY EXPOSURE ASSESSMENT CONSOLE")
            print(" Analyst: Neo | Engine: Pure Python Algorithmic Architecture")
            print("-" * 80)
            print(" [1] Run Complete 16-Step Algorithm (Execute All Calculations & View Results)")
            print(" [2] View Current IoMT Devices & Sensor Readings (Step 1 & 2)")
            print(" [3] View Calculated Reliabilities & Normalised Frequencies (Step 3, 4 & 5)")
            print(" [4] View Security Levels & Security Exposure Scores (Step 6 & 7)")
            print(" [5] View Inter-Device Adjacency & Dependency Matrices (Step 8 & 9)")
            print(" [6] View Network & Edge Reliability (Step 10 & 11: R_network = R_old)")
            print(" [7] View Criticality Ranking & Failure Impact (Step 12 to 15)")
            print(" [8] Modify Device Properties (Sensors, Frequency, Security, Reliability)")
            print(" [9] Manage Inter-Device Dependency Connections (W)")
            print(" [10] Randomize Initial Reliabilities (Within Defined Range [R_min, R_max])")
            print(" [11] Reset Network to Hospital ICU Benchmark (D1-D5)")
            print(" [12] Display Mathematical Formulation & 0-100 Scale Definitions")
            print(" [13] Exit Application")
            print("-" * 80)

            choice = input("Enter your choice [1-13]: ").strip()

            if choice == "1":
                self.action_run_complete_algorithm()
            elif choice == "2":
                self.action_view_devices_and_sensors()
            elif choice == "3":
                self.action_view_reliabilities_and_frequencies()
            elif choice == "4":
                self.action_view_security_exposure()
            elif choice == "5":
                self.action_view_matrices()
            elif choice == "6":
                self.action_view_network_reliability()
            elif choice == "7":
                self.action_view_criticality()
            elif choice == "8":
                self.action_modify_device()
            elif choice == "9":
                self.action_manage_dependencies()
            elif choice == "10":
                self.action_randomize_reliabilities()
            elif choice == "11":
                self.engine.init_default_hospital_network()
                print("\n[+] Network successfully reset to default hospital ICU configuration.")
            elif choice == "12":
                self.action_display_formulation()
            elif choice == "13":
                print("\nExiting IoMT Assessment Console. Goodbye, Neo.\n")
                break
            else:
                print("\n[!] Invalid selection. Please enter a number from 1 to 13.")

    def action_run_complete_algorithm(self):
        print_header("EXECUTING 16-STEP IoMT ALGORITHM")
        results = self.engine.execute_full_algorithm()
        metrics = results["device_metrics"]
        crit = results["criticality_analysis"]

        print(f"\n[*] Maximum Observed Frequency (f_max): {results['f_max']:.2f} Hz")
        print(f"[*] Base Network / Edge Reliability (R_old): {crit['r_old']:.4f} ({crit['r_old_scaled']:.2f} / 100)")

        # Table 1: Device Assessment Summary (Steps 1-7)
        print_subheader("DEVICE ASSESSMENT TABLE (STEPS 1-7)")
        headers = ["ID", "Device Name", "Sensors", "R_init", "R_sensor", "R_device", "Freq(Hz)", "F_norm", "Security", "Exposure(E)"]
        widths = [4, 25, 8, 8, 9, 9, 9, 8, 10, 12]
        rows = []
        for d_id, m in metrics.items():
            rows.append([
                d_id,
                m["name"][:25],
                str(m["sensor_count"]),
                f"{m['r_initial']:.3f}",
                f"{m['r_sensor']:.3f}",
                f"{m['r_device']:.3f}",
                f"{m['frequency_hz']:.1f}",
                f"{m['f_norm']:.3f}",
                f"{m['security_name']}({m['p_val']:.1f})",
                f"{m['exposure_e']:.3f} ({m['exposure_e_100']:.1f}%)"
            ])
        print_table(headers, rows, widths)

        # Table 2: Criticality Ranking (Steps 10-15)
        print_subheader("CRITICALITY ASSESSMENT TABLE (STEPS 10-15)")
        c_headers = ["ID", "Device Name", "R_old", "R_new (Affected)", "Delta R", "Criticality Ratio", "Criticality %", "Clinical Impact"]
        c_widths = [4, 25, 8, 18, 9, 18, 14, 20]
        c_rows = []
        c_table = crit["criticality_table"]
        sorted_devices = sorted(c_table.keys(), key=lambda k: c_table[k]["criticality"], reverse=True)

        for d_id in sorted_devices:
            entry = c_table[d_id]
            impact_tag = "MOST CRITICAL" if d_id == crit["most_critical_device"] else "Secondary Cascade"
            c_rows.append([
                d_id,
                self.engine.devices[d_id].name[:25],
                f"{entry['r_old']:.4f}",
                f"{entry['r_new']:.4f}",
                f"{entry['delta_r']:.4f}",
                f"{entry['criticality']:.4f}",
                f"{entry['criticality_pct']:.2f}%",
                impact_tag
            ])
        print_table(c_headers, c_rows, c_widths)

        print("\n" + "=" * 80)
        print(f" [*] ALGORITHM RESULT: The Most Critical Device is {crit['most_critical_device']} ({crit['most_critical_device_name']})")
        print(f"     Criticality Impact: {crit['max_criticality_pct']:.2f}% network degradation upon failure.")
        print("=" * 80)
        input("\nPress Enter to return to main menu...")

    def action_view_devices_and_sensors(self):
        print_header("IoMT DEVICES & INDEPENDENT SENSOR READINGS (STEPS 1 & 2)")
        headers = ["ID", "Device Name", "Protocol", "Family", "Sensors", "Individual Sensor Readings"]
        widths = [4, 25, 26, 10, 8, 35]
        rows = []
        for d_id in sorted(self.engine.devices.keys()):
            d = self.engine.devices[d_id]
            readings_str = ", ".join(f"{s:.2f}" for s in d.sensor_readings)
            rows.append([d_id, d.name[:25], d.protocol[:26], d.protocol_family, str(d.sensor_count), readings_str])
        print_table(headers, rows, widths)
        input("\nPress Enter to return to main menu...")

    def action_view_reliabilities_and_frequencies(self):
        print_header("RELIABILITY & NORMALIZED FREQUENCY (STEPS 3, 4 & 5)")
        f_max = self.engine.get_max_frequency()
        print(f"Current Maximum Sending Frequency (f_max): {f_max:.2f} Hz\n")
        headers = ["ID", "Device Name", "R_initial", "R_sensor (Step 3)", "R_device (Step 4)", "Freq f", "F = f/f_max (Step 5)"]
        widths = [4, 25, 10, 18, 18, 8, 20]
        rows = []
        for d_id in sorted(self.engine.devices.keys()):
            d = self.engine.devices[d_id]
            r_sens = d.calculate_sensor_reliability()
            r_dev = d.calculate_device_reliability()
            f_norm = d.calculate_normalised_frequency(f_max)
            rows.append([
                d_id, d.name[:25], f"{d.initial_reliability:.3f}",
                f"{r_sens:.4f}", f"{r_dev:.4f} ({r_dev*100:.1f}%)",
                f"{d.frequency_hz:.1f} Hz", f"{f_norm:.4f}"
            ])
        print_table(headers, rows, widths)
        input("\nPress Enter to return to main menu...")

    def action_view_security_exposure(self):
        print_header("SECURITY LEVEL & SECURITY EXPOSURE (STEPS 6 & 7)")
        f_max = self.engine.get_max_frequency()
        headers = ["ID", "Device Name", "Norm Freq (F)", "Security Level", "P Value", "Exposure E = (F+(1-P))/2", "Scale (0-100)"]
        widths = [4, 25, 14, 15, 8, 26, 20]
        rows = []
        for d_id in sorted(self.engine.devices.keys()):
            d = self.engine.devices[d_id]
            f_norm = d.calculate_normalised_frequency(f_max)
            e_val = d.calculate_security_exposure(f_max)
            rows.append([
                d_id, d.name[:25], f"{f_norm:.3f}", d.security_level_name,
                f"{d.security_level_p:.2f}", f"{e_val:.4f}", f"{e_val*100:.1f}%"
            ])
        print_table(headers, rows, widths)
        input("\nPress Enter to return to main menu...")

    def action_view_matrices(self):
        print_header("INTER-DEVICE ADJACENCY & DEPENDENCY MATRICES (STEPS 8 & 9)")
        dev_ids, adj = self.engine.build_adjacency_matrix()
        _, dep = self.engine.build_dependency_matrix()

        print("\n--- Adjacency Matrix A (1 = Connected / Downstream dependency) ---")
        header_row = "       " + "  ".join(f"{d:>5}" for d in dev_ids)
        print(header_row)
        print("       " + "-" * (7 * len(dev_ids)))
        for i, src in enumerate(dev_ids):
            row_str = f"{src:>5} |" + "  ".join(f"{adj[i][j]:>5}" for j in range(len(dev_ids)))
            print(row_str)

        print("\n--- Dependency Weight Matrix W (W_ij = dependency of Target on Source) ---")
        print(header_row)
        print("       " + "-" * (7 * len(dev_ids)))
        for i, src in enumerate(dev_ids):
            row_str = f"{src:>5} |" + "  ".join(f"{dep[i][j]:>5.2f}" for j in range(len(dev_ids)))
            print(row_str)

        print("\nActive Dependency Links:")
        for (src, tgt), w in sorted(self.engine.dependency_weights.items()):
            print(f"  * {src} ({self.engine.devices[src].name}) ---> {tgt} ({self.engine.devices[tgt].name}): Weight W = {w:.2f}")
        input("\nPress Enter to return to main menu...")

    def action_view_network_reliability(self):
        print_header("NETWORK & DERIVED EDGE GATEWAY RELIABILITY (STEPS 10 & 11)")
        r_net = self.engine.calculate_network_reliability()
        print(f" Formula : R_network = (R1 + R2 + ... + Rn) / n")
        print(f" Value   : {r_net:.6f} ({r_net * 100.0:.2f} / 100.0)")
        print(f" Stored  : R_old = {r_net:.6f}")
        print("\n Clinical Constraint Fulfillment:")
        print(" [v] Edge reliability is NOT directly assigned.")
        print(" [v] Edge reliability is derived strictly from all connected medical devices.")
        input("\nPress Enter to return to main menu...")

    def action_view_criticality(self):
        print_header("CRITICALITY ASSESSMENT (STEPS 12 TO 15)")
        crit = self.engine.run_criticality_analysis()
        c_table = crit["criticality_table"]

        headers = ["ID", "Device Name", "R_old", "R_new", "Delta R", "Criticality Ratio", "Criticality %"]
        widths = [4, 25, 8, 8, 9, 18, 14]
        rows = []
        for d_id in sorted(c_table.keys(), key=lambda k: c_table[k]["criticality"], reverse=True):
            entry = c_table[d_id]
            rows.append([
                d_id, self.engine.devices[d_id].name[:25],
                f"{entry['r_old']:.4f}", f"{entry['r_new']:.4f}",
                f"{entry['delta_r']:.4f}", f"{entry['criticality']:.4f}",
                f"{entry['criticality_pct']:.2f}%"
            ])
        print_table(headers, rows, widths)

        print(f"\n[!] Most Critical Device : {crit['most_critical_device']} ({crit['most_critical_device_name']})")
        print(f"[!] Peak Criticality Score: {crit['max_criticality_pct']:.2f}% impact on the entire network.")
        input("\nPress Enter to return to main menu...")

    def action_modify_device(self):
        print_header("MODIFY DEVICE PARAMETERS")
        dev_ids = sorted(self.engine.devices.keys())
        target_id = input(f"Enter Device ID to modify ({', '.join(dev_ids)}): ").strip().upper()

        if target_id not in self.engine.devices:
            print(f"[!] Device {target_id} not found.")
            return

        d = self.engine.devices[target_id]
        print(f"\nModifying {d.device_id} ({d.name}):")

        new_r = input(f"Enter new Initial Reliability [0.0 - 1.0] (Enter to keep {d.initial_reliability}): ").strip()
        if new_r:
            try:
                val = float(new_r)
                if 0.0 <= val <= 1.0:
                    d.initial_reliability = val
            except ValueError:
                pass

        new_f = input(f"Enter new Frequency in Hz (Enter to keep {d.frequency_hz}): ").strip()
        if new_f:
            try:
                val = float(new_f)
                if val > 0:
                    d.frequency_hz = val
            except ValueError:
                pass

        print("Security Options: [1] Strong (0.9)  [2] Medium (0.7)  [3] Weak (0.4)")
        sec_choice = input("Select Security Level [1-3] (Enter to keep): ").strip()
        if sec_choice == "1":
            d.security_level_name = "Strong"
            d.security_level_p = 0.9
        elif sec_choice == "2":
            d.security_level_name = "Medium"
            d.security_level_p = 0.7
        elif sec_choice == "3":
            d.security_level_name = "Weak"
            d.security_level_p = 0.4

        new_sensors_raw = input("Enter comma-separated sensor readings (>= 2 values) (Enter to keep): ").strip()
        if new_sensors_raw:
            try:
                readings = [float(x.strip()) for x in new_sensors_raw.split(",") if x.strip()]
                if len(readings) >= 2:
                    d.sensor_readings = readings
                else:
                    print("  [!] Violation: Must provide >= 2 sensors. Kept previous.")
            except ValueError:
                pass

        print(f"[+] Device {target_id} updated successfully.")
        input("\nPress Enter to return to main menu...")

    def action_manage_dependencies(self):
        print_header("MANAGE INTER-DEVICE DEPENDENCY CONNECTIONS (W)")
        for (src, tgt), w in sorted(self.engine.dependency_weights.items()):
            print(f"  {src} -> {tgt} : Weight = {w:.2f}")

        print("\nOptions: [1] Add / Update Connection  [2] Remove Connection  [3] Back")
        c = input("Choice [1-3]: ").strip()
        if c == "1":
            src = input("Enter Source Device ID: ").strip().upper()
            tgt = input("Enter Target Device ID: ").strip().upper()
            if src in self.engine.devices and tgt in self.engine.devices and src != tgt:
                try:
                    w = float(input("Enter dependency weight W [0.01 - 1.0]: ").strip())
                    if 0.0 < w <= 1.0:
                        self.engine.set_dependency(src, tgt, w)
                        print(f"  [+] Set connection {src} -> {tgt} with weight {w:.2f}")
                except ValueError:
                    pass
        elif c == "2":
            src = input("Enter Source Device ID: ").strip().upper()
            tgt = input("Enter Target Device ID: ").strip().upper()
            self.engine.dependency_weights.pop((src, tgt), None)
            print(f"  [+] Removed connection {src} -> {tgt}")
        input("\nPress Enter to return to main menu...")

    def action_randomize_reliabilities(self):
        print_header("RANDOMIZE INITIAL RELIABILITIES")
        r_min_str = input("Enter minimum initial reliability (default 0.70): ").strip() or "0.70"
        r_max_str = input("Enter maximum initial reliability (default 0.95): ").strip() or "0.95"
        try:
            r_min = float(r_min_str)
            r_max = float(r_max_str)
            if 0.0 <= r_min <= r_max <= 1.0:
                for d in self.engine.devices.values():
                    d.initial_reliability = round(random.uniform(r_min, r_max), 3)
                print(f"[+] Randomized initial reliabilities in range [{r_min:.2f}, {r_max:.2f}].")
        except ValueError:
            print("[!] Invalid numerical input.")
        input("\nPress Enter to return to main menu...")

    def action_display_formulation(self):
        print_header("MATHEMATICAL FORMULATIONS & 0-100 SCALE SPECIFICATION")
        print("""
* Step 3: Sensor Reliability:
    R_sensor = (sum of sensor values) / (number of sensors)   [sensors >= 2]

* Step 4: Device Reliability:
    R_device = (R_initial + R_sensor) / 2

* Step 5: Normalised Communication Frequency:
    F = f / f_max

* Step 6 & 7: Security Exposure:
    P in {Strong: 0.9, Medium: 0.7, Weak: 0.4}
    Exposure E = (F + (1 - P)) / 2

* Step 10 & 11: Network / Derived Edge Reliability:
    R_network = (R_1 + R_2 + ... + R_n) / n = R_old

* Step 12 to 15: Criticality & Cascade:
    When D_k is affected (R_k -> 0.0):
      Dependent devices degrade: R_j_new = R_j * (1 - W_kj)
      R_new = (R_1_affected + ... + R_n_affected) / n
      Criticality_k = (R_old - R_new) / R_old
    Device with max(Criticality) is the Critical Device.
        """)
        input("\nPress Enter to return to main menu...")


# ==============================================================================
# MAIN ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    app = IoMTAlgorithmApp()
    app.run()


            IoMT RELIABILITY & SECURITY EXPOSURE ASSESSMENT CONSOLE             
 Analyst: Neo | Engine: Pure Python Algorithmic Architecture
--------------------------------------------------------------------------------
 [1] Run Complete 16-Step Algorithm (Execute All Calculations & View Results)
 [2] View Current IoMT Devices & Sensor Readings (Step 1 & 2)
 [3] View Calculated Reliabilities & Normalised Frequencies (Step 3, 4 & 5)
 [4] View Security Levels & Security Exposure Scores (Step 6 & 7)
 [5] View Inter-Device Adjacency & Dependency Matrices (Step 8 & 9)
 [6] View Network & Edge Reliability (Step 10 & 11: R_network = R_old)
 [7] View Criticality Ranking & Failure Impact (Step 12 to 15)
 [8] Modify Device Properties (Sensors, Frequency, Security, Reliability)
 [9] Manage Inter-Device Dependency Connections (W)
 [10] Randomize Initial Reliabilities (Within Defined Range [R_min, R_max])
 [11] Reset Network to Hospital ICU Benchmark (D1-D5)
 [12] Display Mathematical Formulati